# 04 · The pipeline, pass by pass

`tvm.compile(mod, target, relax_pipeline=...)` runs a Relax pipeline, then generates code. Production
CUDA builds use `opt_fast_dllm_llm_cublas` (`fast_dllm_relax/model/cuda.py`).

This notebook runs that pipeline, unmodified, on the tiny fp16 model under an **`IRRecorder`**: a TVM
`PassInstrument` that captures the IRModule after every pass, including the passes the pipeline calls
internally. Each section then zooms in on one stage:
- (a) BYOC offload to cuBLAS and CUTLASS;
- (b) `LegalizeOps`, from ops to TIR;
- (c) fusion;
- (d) dlight scheduling;
- (e) what changes for bf16 and for the Jetson build, which has no CUTLASS.

**Needs:** CUDA and a TVM build with cuBLAS and CUTLASS. About a minute (CUTLASS's nvcc build is most
of it). Every module is also written in full to `docs/notebooks/_dumps/04_fp16/` (gitignored), for
reading in an editor.

In [1]:
from _helpers import *
bootstrap()

repo root: fast-dllm-relax/  (cwd set)  tvm 0.24.dev0  torch 2.10.0+cu128


PosixPath('<repo>')

In [2]:
import tvm
from tvm import relax
from tvm.s_tir import dlight

model, mod, named = export_model(tiny_config(tied=True), dtype="float16")
dev = tvm.cuda(0)
target = tvm.target.Target.from_device(dev)
pipeline = relax.get_pipeline("opt_fast_dllm_llm_cublas", use_cutlass=True, verbose=False)

rec = make_recorder(dump=True, run_name="04_fp16")
lowered = lower(mod, target, pipeline, recorder=rec)
pass_table(rec, max_depth=0)

[16:49:28] <repo>/3rdparty/tvm/include/tvm/topi/transform.h:1253: Warning: Fast mode segfaults when there are out-of-bounds indices. Make sure input indices are in bound


#     pass                    ms        relax_fns  tir_fns  codegen_fns  calls  call_tir  extern_calls
----  ----------------------  --------  ---------  -------  -----------  -----  --------  ------------
0     FuseOpsByPattern        84.9      17         0        7            1416   0         12          
1     FuseOpsByPattern        505.6     18         0        8            1418   0         12          
2     RunCodegen              1.05e+04  10         0        0            1353   0         77          
4     FuseTransposeMatmul     35        10         0        0            1353   0         77          
5     LegalizeOps             284.5     10         125      0            1329   1252      77          
6     AnnotateTIROpPattern    0.4       10         125      0            1329   1252      77          
3847  FoldConstant            2336      10         125      0            1265   1188      77          
3848  FuseOps                 27.7      58         125      0            

Reading the table:
- `relax_fns` = Relax functions, `tir_fns` = TIR PrimFuncs (kernels), `codegen_fns` = functions
  handed to a BYOC backend.
- `calls` counts every call inside Relax functions; `call_tir` and `extern_calls` are subsets of it.
- Use `pass_table(rec)` without `max_depth` to also see the passes each pass runs internally
  (FuseOps and FuseTIR run several).

The stages:

| Stage | Passes | What happens |
|---|---|---|
| BYOC | `FuseOpsByPattern` ×2, `RunCodegen` | GEMMs matching cuBLAS/CUTLASS patterns become external library calls |
| Lowering | `FuseTransposeMatmul`, `LegalizeOps` | Remaining high-level ops become TIR PrimFuncs |
| Fusion | `AnnotateTIROpPattern`, `FoldConstant`, `FuseOps`, `FuseTIR`, `DCE` | Chains of elementwise/injective kernels merge into one |
| Scheduling | `ApplyDefaultSchedule` (dlight) | Each PrimFunc gets a GPU thread/block schedule |
| VM lowering | `RewriteDataflowReshape` … `AttachExternModules` | Memory planning and VM-level builtins (notebook 05) |

## (a) BYOC: cuBLAS first, then CUTLASS

In [3]:
after_cublas = rec.after("FuseOpsByPattern", 0)
cublas_fns = find_functions(after_cublas, codegen="cublas")
print(f"{len(cublas_fns)} functions offloaded to cuBLAS, e.g. {cublas_fns[:4]}")
show_ir(after_cublas, cublas_fns[0], max_lines=25)

7 functions offloaded to cuBLAS, e.g. ['fused_relax_permute_dims_relax_matmul1_cublas', 'fused_relax_permute_dims_relax_matmul2_cublas', 'fused_relax_permute_dims_relax_matmul3_cublas', 'fused_relax_permute_dims_relax_matmul4_cublas']
# from tvm.script import tirx as T
# from tvm.script import relax as R

@R.function
def fused_relax_permute_dims_relax_matmul1_cublas(model_layers_0_mlp_gate_up_proj_weight1: R.Tensor((1024, 256), dtype="float16"), mul10: R.Tensor((1, "seq_len", 256), dtype="float16")) -> R.Tensor((1, "seq_len", 1024), dtype="float16"):
    seq_len = T.int64()
    R.func_attr({"Codegen": "cublas"})
    # from tvm.script import tirx as T
    # from tvm.script import relax as R
    
    @R.function
    def local_func(model_layers_0_mlp_gate_up_proj_weight1_1: R.Tensor((1024, 256), dtype="float16"), mul10_1: R.Tensor((1, seq_len, 256), dtype="float16")) -> R.Tensor((1, seq_len, 1024), dtype="float16"):
        R.func_attr({"Composite": "cublas.matmul_transposed"})
        wi

Each offloaded GEMM becomes a small Relax function tagged `Codegen="cublas"`. Inside it is a
`Composite` function (the matched pattern: matmul, plus bias add when there is one). The entry
function now calls these functions instead of `R.matmul`:

In [4]:
show_ir(after_cublas, "forward_hidden", grep=r"cublas|R\.matmul", max_lines=20)

        lv: R.Tensor((1, seq_len, 1024), dtype="float16") = fused_relax_permute_dims_relax_matmul_relax_add_cublas(model_layers_0_self_attn_qkv_proj_weight1, mul1, model_layers_0_self_attn_qkv_proj_bias1)
    ...
        matmul1: R.Tensor((1, 4, seq_len, seq_len), dtype="float32") = R.matmul(permute_dims1, permute_dims2, out_dtype="float32")
    ...
        matmul2: R.Tensor((1, 4, seq_len, 128), dtype="float32") = R.matmul(softmax, astype2, out_dtype="float32")
    ...
        lv_2: R.Tensor((1, seq_len, 256), dtype="float16") = fused_relax_permute_dims_relax_matmul_cublas(model_layers_0_self_attn_o_proj_weight1, reshape3)
    ...
        lv1_1: R.Tensor((1, seq_len, 1024), dtype="float16") = fused_relax_permute_dims_relax_matmul1_cublas(model_layers_0_mlp_gate_up_proj_weight1, mul10)
    ...
        lv2_1: R.Tensor((1, seq_len, 256), dtype="float16") = fused_relax_permute_dims_relax_matmul_cublas(model_layers_0_mlp_down_proj_weight1, mul12)
    ...
        lv1_2: R.Tensor((1, seq_len

In [5]:
after_cutlass = rec.after("FuseOpsByPattern", 1)
cutlass_fns = find_functions(after_cutlass, codegen="cutlass")
print(f"{len(cutlass_fns)} functions offloaded to CUTLASS: {cutlass_fns[:6]}")
if cutlass_fns:
    show_ir(after_cutlass, cutlass_fns[0], max_lines=20)

1 functions offloaded to CUTLASS: ['fused_relax_matmul_cutlass']
# from tvm.script import tirx as T
# from tvm.script import relax as R

@R.function
def fused_relax_matmul_cutlass(permute_dims1: R.Tensor((1, 4, "seq_len", 128), dtype="float16"), permute_dims2: R.Tensor((1, 4, 128, "seq_len"), dtype="float16")) -> R.Tensor((1, 4, "seq_len", "seq_len"), dtype="float32"):
    seq_len = T.int64()
    R.func_attr({"Codegen": "cutlass"})
    # from tvm.script import tirx as T
    # from tvm.script import relax as R
    
    @R.function
    def local_func(permute_dims1_1: R.Tensor((1, 4, seq_len, 128), dtype="float16"), permute_dims2_1: R.Tensor((1, 4, 128, seq_len), dtype="float16")) -> R.Tensor((1, 4, seq_len, seq_len), dtype="float32"):
        R.func_attr({"Composite": "cutlass.matmul"})
        with R.dataflow():
            gv: R.Tensor((1, 4, seq_len, seq_len), dtype="float32") = R.matmul(permute_dims1_1, permute_dims2_1, out_dtype="float32")
            R.output(gv)
        return gv


CUTLASS takes the **batched** attention matmuls (`Q·Kᵀ`, `P·V`, 4-D × 4-D), which cuBLAS's pattern
rejects. The order matters: CUTLASS also has a generic dense pattern, so running it first would
claim the projection GEMMs and leave cuBLAS with nothing. Projections and `get_logits` therefore stay
on cuBLAS, the library PyTorch uses as well.

`RunCodegen` then compiles each tagged function into an external runtime module and replaces the
call with a call to an extern symbol:

In [6]:
after_codegen = rec.after("RunCodegen")
s = ir_stats(after_codegen)
print("external runtime modules attached to the IRModule:", s["external_mods"])
print("extern calls by symbol (top 8):")
print_table([{"symbol": k, "calls": v} for k, v in s["externs"].most_common(8)])

external runtime modules attached to the IRModule: 8
extern calls by symbol (top 8):
symbol                                                   calls
-------------------------------------------------------  -----
fused_relax_permute_dims_relax_matmul3_cublas            16   
fused_relax_permute_dims_relax_matmul_cublas             14   
fast_dllm.flash_attn_bf16                                12   
fused_relax_permute_dims_relax_matmul_relax_add_cublas   8    
fused_relax_permute_dims_relax_matmul_relax_add1_cublas  8    
fused_relax_permute_dims_relax_matmul4_cublas            8    
fused_relax_permute_dims_relax_matmul1_cublas            7    
fused_relax_matmul_cutlass                               3    


## (b) `LegalizeOps`: high-level ops become TIR

Every remaining Relax op (`R.add`, `R.mean`, `R.astype`, the explicit attention matmuls in bf16, ...)
becomes an `R.call_tir` to a generated PrimFunc. Here is RMSNorm's mean-of-squares as TIR:

In [7]:
legal = rec.after("LegalizeOps")
b, a = ir_stats(rec.before("LegalizeOps")), ir_stats(legal)
print(f"call_tir: {b['call_tir']} -> {a['call_tir']}   PrimFuncs: {b['tir_fns']} -> {a['tir_fns']}")
mean_fns = find_functions(legal, r"mean", kind="tir")
show_ir(legal, mean_fns[0], max_lines=30)

call_tir: 0 -> 1252   PrimFuncs: 0 -> 125
# from tvm.script import tirx as T

@T.prim_func(private=True)
def main(var_square5: T.handle, var_T_divide: T.handle):
    T.func_attr({"tirx.noalias": True})
    prefix_len = T.int64()
    square5 = T.match_buffer(var_square5, (T.int64(1), prefix_len, T.int64(256)))
    T_divide = T.match_buffer(var_T_divide, (T.int64(1), prefix_len, T.int64(1)))
    # with T.sblock("root"):
    square5_red = T.sblock_alloc_buffer((T.int64(1), prefix_len, T.int64(1)))
    for ax0, ax1, ax2, k2 in T.grid(T.int64(1), prefix_len, T.int64(1), T.int64(256)):
        with T.sblock("square5_red"):
            v_ax0, v_ax1, v_ax2, v_k2 = T.axis.remap("SSSR", [ax0, ax1, ax2, k2])
            T.reads(square5[v_ax0, v_ax1, v_k2])
            T.writes(square5_red[v_ax0, v_ax1, v_ax2])
            with T.init():
                square5_red[v_ax0, v_ax1, v_ax2] = T.float32(0.0)
            square5_red[v_ax0, v_ax1, v_ax2] = square5_red[v_ax0, v_ax1, v_ax2] + square5[v_ax0,

## (c) Fusion: chains of kernels merge

`FuseOps` groups neighbouring kernels by their op pattern (elementwise, broadcast, injective,
reduction), and `FuseTIR` merges each group into one PrimFunc. In this model that fuses, for example:
- RMSNorm's cast → multiply → cast → weight-multiply;
- the fp32 SiLU · up product;
- RoPE's multiply-add.

In [8]:
fused = rec.after("FuseTIR")
fused_fns = find_functions(fused, r"^fused_", kind="tir")
print(f"PrimFuncs: {ir_stats(legal)['tir_fns']} after LegalizeOps -> {ir_stats(fused)['tir_fns']} after FuseTIR "
      f"({len(fused_fns)} named fused_*)")
print("\n".join(fused_fns[:20]))

PrimFuncs: 125 after LegalizeOps -> 69 after FuseTIR (48 named fused_*)
fused_add4_cast
fused_add9_cast8
fused_arange_cast10_add10_expand_dims_multiply3_concatenate
fused_arange_cast2_expand_dims_multiply3_concatenate
fused_cast10_add6_expand_dims5_multiply13_concatenate4
fused_expand_dims10_expand_dims10_concatenate15
fused_expand_dims4_expand_dims4_concatenate3
fused_expand_dims8_expand_dims8_concatenate9
fused_expand_dims9_expand_dims9_concatenate10
fused_matmul_cast5
fused_mean1_add5_tir_rsqrt1_multiply11_cast9_multiply12
fused_mean_add_tir_rsqrt_multiply_cast1_multiply1
fused_multiply4_split1_multiply5_concatenate1_multiply4_add1
fused_multiply4_split1_multiply5_concatenate1_multiply4_add1_transpose1
fused_multiply6_split2_multiply7_concatenate2_multiply6_add2
fused_multiply6_split2_multiply7_concatenate2_multiply6_add2_repeat_transpose2
fused_multiply6_split2_multiply7_concatenate2_multiply6_add2_transpose
fused_multiply8_add3
fused_repeat_transpose1_cast4
fused_repeat_transpose2

In [9]:
silu = [f for f in fused_fns if "sigmoid" in f]
show_ir(fused, silu[0] if silu else fused_fns[0], max_lines=40)

# from tvm.script import tirx as T

@T.prim_func(private=True)
def main(p_lv44: T.handle, p_output0: T.handle):
    T.func_attr({"tirx.noalias": True})
    prefix_len = T.int64()
    lv44 = T.match_buffer(p_lv44, (T.int64(1), prefix_len, T.int64(1024)), "float16")
    T_multiply_intermediate_1 = T.match_buffer(p_output0, (T.int64(1), prefix_len, T.int64(512)), "float16")
    # with T.sblock("root"):
    T_split_sections_intermediate = T.sblock_alloc_buffer((T.int64(1), prefix_len, T.int64(512)), "float16")
    T_split_sections_intermediate_1 = T.sblock_alloc_buffer((T.int64(1), prefix_len, T.int64(512)), "float16")
    compute_intermediate = T.sblock_alloc_buffer((T.int64(1), prefix_len, T.int64(512)))
    compute_intermediate_1 = T.sblock_alloc_buffer((T.int64(1), prefix_len, T.int64(512)))
    T_multiply_intermediate = T.sblock_alloc_buffer((T.int64(1), prefix_len, T.int64(512)))
    compute_intermediate_1_2 = T.sblock_alloc_buffer((T.int64(1), prefix_len, T.int64(512)), "float16")
 

## (d) dlight: one rule per kernel

`ApplyDefaultSchedule` tries each rule in order and uses the first one that accepts the PrimFunc. The
production rule list is `Matmul, GEMV, Reduction, GeneralReduction, Fallback`. `dlight_rule_table`
reruns that choice on the module just before scheduling:

In [10]:
rules = [dlight.gpu.Matmul(), dlight.gpu.GEMV(), dlight.gpu.Reduction(),
         dlight.gpu.GeneralReduction(), dlight.gpu.Fallback()]
pre_sched = rec.before("ApplyDefaultSchedule")
picked = dlight_rule_table(pre_sched, target, rules)
print_table([{"prim_func": k, "rule": v} for k, v in sorted(picked.items())][:25])

rule                 prim_funcs
-------------------  ----------
(already scheduled)  69        
prim_func                                                                       rule               
------------------------------------------------------------------------------  -------------------
add4                                                                            (already scheduled)
add9                                                                            (already scheduled)
cast                                                                            (already scheduled)
cast14                                                                          (already scheduled)
cast8                                                                           (already scheduled)
concatenate14                                                                   (already scheduled)
concatenate8                                                                    (already scheduled)
fuse

Here is one PrimFunc before and after scheduling. Scheduling adds thread and block bindings and
shared-memory staging; the arithmetic stays the same.

In [11]:
name = next((k for k, v in picked.items() if v == "GeneralReduction"), next(iter(picked)))
print("##", name, "->", picked[name])
show_ir(pre_sched, name, max_lines=25)
print("\n## after ApplyDefaultSchedule")
show_ir(rec.after("ApplyDefaultSchedule"), name, max_lines=45)

## add4 -> (already scheduled)
# from tvm.script import tirx as T

@T.prim_func(private=True)
def main(var_prefix_embed: T.handle, var_lv43: T.handle, var_T_add: T.handle):
    T.func_attr({"op_pattern": 0, "tirx.is_scheduled": True, "tirx.noalias": True})
    prefix_len = T.int64()
    prefix_embed = T.match_buffer(var_prefix_embed, (T.int64(1), prefix_len, T.int64(256)), "float16")
    lv43 = T.match_buffer(var_lv43, (T.int64(1), prefix_len, T.int64(256)), "float16")
    T_add = T.match_buffer(var_T_add, (T.int64(1), prefix_len, T.int64(256)), "float16")
    # with T.sblock("root"):
    for ax0_ax1_fused_0 in T.thread_binding((prefix_len * T.int64(256) + T.int64(1023)) // T.int64(1024), thread="blockIdx.x"):
        for ax0_ax1_fused_1 in T.thread_binding(T.int64(1024), thread="threadIdx.x"):
            with T.sblock("T_add"):
                v0 = T.axis.spatial(prefix_len, (ax0_ax1_fused_0 * T.int64(1024) + ax0_ax1_fused_1) // T.int64(256))
                v1 = T.axis.spatial(T.int

## (e) bf16, and the Jetson configuration without CUTLASS

CUTLASS's batched-GEMM pattern does not claim the **bf16** attention matmuls. Those matmuls only
exist in the non-cached paths (`forward_hidden`, `build_prefix_cache`); the cached paths use the
PyTorch callback. So which dlight rule schedules them?

Two more lowerings answer that:
- bf16 with CUTLASS on: the production bf16 artifact.
- fp16 with `use_cutlass=False`: the Jetson build, whose rule list drops `Matmul()` because it hangs on
  these mixed-precision batched matmuls.

In [12]:
import collections

def n_codegen(r, backend):
    hits = [x for x in r.records if x.name == "FuseOpsByPattern"]
    return max((len(find_functions(x.mod, codegen=backend)) for x in hits), default=0)

def lower_variant(dtype, use_cutlass):
    _, m, _ = export_model(tiny_config(tied=True), dtype=dtype)
    r = make_recorder(keep_modules=True)
    lower(m, target, relax.get_pipeline("opt_fast_dllm_llm_cublas", use_cutlass=use_cutlass, verbose=False), recorder=r)
    return r

variants = {"fp16 + CUTLASS": (rec, rules),
            "bf16 + CUTLASS": (lower_variant("bfloat16", True), rules),
            "fp16, no CUTLASS (Jetson)": (lower_variant("float16", False), rules[1:])}
rows = []
for label, (r, rl) in variants.items():
    pre = r.before("ApplyDefaultSchedule")
    pick = dlight_rule_table(pre, target, rl, show=False)
    attn = {k: v for k, v in pick.items() if "matmul" in k}
    rows.append({"variant": label,
                 "cublas fns": n_codegen(r, "cublas"),
                 "cutlass fns": n_codegen(r, "cutlass"),
                 "TIR matmul kernels": len(attn),
                 "their rules": dict(collections.Counter(attn.values()))})
print_table(rows)

[16:49:34] <repo>/3rdparty/tvm/include/tvm/topi/transform.h:1253: Warning: Fast mode segfaults when there are out-of-bounds indices. Make sure input indices are in bound
[16:49:37] <repo>/3rdparty/tvm/include/tvm/topi/transform.h:1253: Warning: Fast mode segfaults when there are out-of-bounds indices. Make sure input indices are in bound


variant                    cublas fns  cutlass fns  TIR matmul kernels  their rules               
-------------------------  ----------  -----------  ------------------  --------------------------
fp16 + CUTLASS             7           1            1                   {'(already scheduled)': 1}
bf16 + CUTLASS             7           0            2                   {'(already scheduled)': 2}
fp16, no CUTLASS (Jetson)  7           0            2                   {'(already scheduled)': 2}


In the bf16 and no-CUTLASS variants, the batched attention matmuls stay as TIR kernels. The table
shows which rule takes them. That rule is what schedules the bf16 artifact's prefill attention
(research 001, open question 2), and notebook 07 measures what it costs.

## Takeaways
- The pipeline is **BYOC first**: pattern matching on the high-level IR, then compiling the claimed
  subgraphs into external modules. Only after that does the remaining graph get lowered, fused and
  scheduled by TVM itself.
- Pattern **order** decides which library gets a GEMM (cuBLAS before CUTLASS).
- Every non-library kernel is a PrimFunc with exactly one dlight rule; `dlight_rule_table` shows which.
- Next: **05 · Memory planning and VM lowering**.